# Qwen3-VL-32B LoRA 학습 실험

이 노트북은 `Qwen/Qwen3-VL-32B-Instruct` 학습 실험을 담고 있습니다. 파일명은 `0.9333`이지만, 실제 제출 점수는 `0.93141`입니다.

- 학습 프레임워크: Unsloth, TRL, PEFT LoRA
- 목적: 8B 실험 이후 32B로 확장한 학습 파이프라인 확인
- 산출물: answer와 a/b/c/d 확률 저장


## 1. 환경 준비

In [3]:
# colab & local PC용 라이브러리 설치
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.0
!pip install --no-deps trl==0.22.2
!pip install wandb scikit-learn albumentations # 추가 라이브러리
!pip install ninja
# Blackwell 전용, CPU 풀가동 빌드
# 1. 빌드된 결과물을 휠 파일로 저장 (최초 1회)
# !MAX_JOBS=$(nproc) TORCH_CUDA_ARCH_LIST="10.0" pip wheel flash-attn --no-build-isolation -v -w /content/drive/MyDrive/my_wheels/
# 2. 다음 세션에서 설치할 때 (빛의 속도)
# !pip install /content/drive/MyDrive/my_wheels/flash_attn-*.whl

## 2. 데이터 준비 및 Wandb 로그인

In [1]:
# 구글드라이브 마운트
from google.colab import drive, userdata
import wandb
import os

drive.mount('/content/drive')
%cd '/content/drive/MyDrive/Colab Notebooks'


# Colab Secrets에서 키를 가져와 환경 변수로 설정
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_KEY')

# 자동으로 환경 변수를 읽어 로그인 시도
wandb.login()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: qaz000219 (qaz000219-seoul-national-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 3. 라이브러리, 데이터, 설정 로드

### 데이터 로드와 메타데이터 생성
이 긴 셀은 CSV를 읽고 경로를 정리한 뒤, 질문 유형/재질 분류 함수를 이용해 학습용 메타데이터를 붙입니다.
뒤 학습과 검증, 샘플링, 프롬프트 생성이 모두 여기서 만든 데이터 구조를 기준으로 진행됩니다.

In [2]:
import os, random
import pandas as pd
from PIL import Image
import torch
from datasets import Dataset, DatasetDict
from collections import Counter
from unsloth import FastVisionModel
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import get_chat_template
from unsloth.trainer import UnslothVisionDataCollator
from tqdm import tqdm
import math
import numpy as np
import albumentations as A
from sklearn.utils import resample

SEED = 42
RUN_NAME = "(valid)qwen3_v2_32B_epoch5_standard"
SAVE_DIR = f"run_outputs/{RUN_NAME}"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

random.seed(SEED)
np.random.seed(SEED)  # ← 이 줄 추가
torch.manual_seed(SEED)

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 설정값 정의
MODEL_ID = "Qwen/Qwen3-VL-32B-Instruct"  # 플랜A: 8B → 32B 변경
MAX_SEQ_LENGTH = 768
MAX_NEW_TOKENS = 8
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ===== 경로 설정 (환경에 맞게 수정) =====
DATA_DIR = "."
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
DEV_CSV = os.path.join(DATA_DIR, "dev.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
IMAGE_ROOT = DATA_DIR  # csv의 path 컬럼과 결합될 루트 경로

# 데이터셋 로드
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

# dev.csv 로드 + 다수결 처리 (answer1~5 → answer)
dev_df = pd.read_csv(DEV_CSV)
print(f"원본 데이터 — Train: {len(train_df)}, Dev: {len(dev_df)}, Test: {len(test_df)}")


# ==========================================
# [신규] 질문 유형 / 재질 클래스 분류 + Train 오버샘플링
# ==========================================

def classify_question_type(question):
    q = str(question).lower()
    if any(w in q for w in ['몇 개', '개수', '얼마나', '몇개']):
        return 'count'
    elif any(w in q for w in ['재질', '무엇으로', '만들어진', '어떤 소재']):
        return 'material'
    elif any(w in q for w in ['색상', '무슨 색', '색깔', '어떤 색']):
        return 'color'
    elif any(w in q for w in ['분리수거', '재활용 분류', '분류 기호', '분리배출', '어떻게 배출']):
        return 'recycling_class'
    elif any(w in q for w in ['가장 많이', '가장 많은', '주로', '주된', '가장 큰 비중', '비율']):
        return 'dominant_or_majority'
    elif any(w in q for w in ['위치', '어디에', '왼쪽', '오른쪽', '위', '아래', '옆', '어느 쪽에']):
        return 'position'
    elif any(w in q for w in ['브랜드', '상표', '로고', '제품명']):
        return 'brand_or_product'
    elif any(w in q for w in ['맛']):
        return 'flavor'
    elif any(w in q for w in ['글자', '기호', '텍스트', '쓰여진', '적혀있는', '적힌', '마크']):
        return 'text_or_symbol'
    elif any(w in q for w in ['종류', '무엇인가요', '품목', '무엇입니까', '어떤']):
        return 'object_type'
    else:
        return 'other'

def assign_material_class(row):
    text = str(row['question']) + " " + str(row.get('answer', ''))
    for kw in ['스티로폼', '유리', '캔', '종이', '비닐', '플라스틱']:
        if kw in text:
            return kw
    return '기타'

for df in [train_df, dev_df, test_df]:
    df['q_type'] = df['question'].apply(classify_question_type)

train_df['material_class'] = train_df.apply(assign_material_class, axis=1)

# ===== Train 오버샘플링 =====
print("=" * 50)
print("========== [Train] 데이터 처리 ==========")
print("=" * 50)

print(f"\n[Train] 증강 전 전체 데이터 개수: {len(train_df)}개")
print(f"[Train] 증강 전 고유 ID 개수: {train_df['id'].nunique()}개")

before_counts = train_df['material_class'].value_counts()
target_count = before_counts.max()
MAX_MULTIPLIER = 3.0

augmented_rows = []

print(f"\n=== 클래스별 증강 내역 ===")
for m_class, count in before_counts.items():
    class_df = train_df[train_df['material_class'] == m_class].copy()
    class_df['apply_aug'] = False
    augmented_rows.append(class_df)

    if m_class == '기타':
        print(f"- {m_class}: 원본 {count}개 -> 최종 {count}개")
        continue

    max_allowed = int(count * MAX_MULTIPLIER)
    final_target = min(target_count, max_allowed)
    shortfall = final_target - count

    if shortfall > 0:
        oversampled = resample(
            class_df, replace=True,
            n_samples=shortfall, random_state=SEED
        )
        oversampled['apply_aug'] = True
        augmented_rows.append(oversampled)
        print(f"- {m_class}: 원본 {count}개 -> 최종 {count + shortfall}개")
    else:
        print(f"- {m_class}: 원본 {count}개 -> 최종 {count}개")

train_metadata = pd.concat(augmented_rows).reset_index(drop=True)

print(f"\n[Train] 증강 완료 후 전체 데이터 개수: {len(train_metadata)}개")
print(f"[Train] 증강 완료 후 고유 ID 개수: {train_metadata['id'].nunique()}개 (원본과 정확히 일치해야 함)")
assert train_metadata['id'].nunique() == train_df['id'].nunique(), \
    "❌ 오류: 증강 후 고유 ID 수가 원본과 다릅니다!"
print("✅ 고유 ID 무결성 검증 통과")

n_original = int((~train_metadata['apply_aug']).sum())
n_augmented = int(train_metadata['apply_aug'].sum())
print(f"\n[Train] 원본(증강X): {n_original}개, 복제본(증강O): {n_augmented}개")

print(f"\n=== 증강 후 material_class 분포 ===")
print(train_metadata['material_class'].value_counts().to_string())

print(f"\n=== train_metadata DataFrame 구조 ===")
print(f"shape: {train_metadata.shape}")
print(f"columns: {list(train_metadata.columns)}")
print(f"dtypes:\n{train_metadata.dtypes.to_string()}")
preview = ['id', 'question', 'answer', 'material_class', 'q_type', 'apply_aug']
print(f"\n처음 3행:\n{train_metadata[[c for c in preview if c in train_metadata.columns]].head(3).to_string()}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Device: cuda
원본 데이터 — Train: 5073, Dev: 4413, Test: 5074
========== [Train] 데이터 처리 ==========

[Train] 증강 전 전체 데이터 개수: 5073개
[Train] 증강 전 고유 ID 개수: 5073개

=== 클래스별 증강 내역 ===
- 기타: 원본 1866개 -> 최종 1866개
- 플라스틱: 원본 1835개 -> 최종 1866개
- 종이: 원본 650개 -> 최종 1866개
- 캔: 원본 455개 -> 최종 1365개
- 유리: 원본 160개 -> 최종 480개
- 비닐: 원본 75개 -> 최종 225개
- 스티로폼: 원본 32개 -> 최종 96개

[Train] 증강 완료 후 전체 데이터 개수: 7764개
[Train] 증강 완료 후 고유 ID 개수: 5073개 (원본과 정확히 일치해야 함)
✅ 고유 ID 무결성 검증 통과

[Train] 원본(증강X): 5073개, 복제본(증강O): 2691개

=== 증강 후 material_class 분포 ===
material_class
기타      1866
플라스틱    1866
종이      1866
캔       1365
유리       480
비닐       225
스티로폼      96

=== train_metadata DataFrame 구조 ===
shape: (7764, 11)
columns: ['id', 'path', 'question', 'a', 'b', 'c', 'd', 'answer', 'q_type', 'material_class', 'apply_aug']
dtypes:
id                object
path              object
questio

### Dev 기반 검증 세트 구성
이 구간은 dev 다중 응답을 pseudo-label로 바꾸고, 일치율 필터를 적용해 검증용 데이터셋 후보를 만듭니다.
train만 쓰는 것이 아니라 dev를 정제해 validation으로 쓰려는 의도가 드러나는 부분입니다.

In [3]:
# ==========================================
# [신규] Dev 데이터: pseudo-label + 일치율 필터링 + test 비율 샘플링
# ==========================================

print("\n" + "=" * 50)
print("========== [Valid] 데이터 처리 ==========")
print("=" * 50)

print(f"\n[Valid] 초기 Dev 전체 데이터 개수: {len(dev_df)}개")
print(f"[Valid] 초기 Dev 고유 ID 개수: {dev_df['id'].nunique()}개")

response_cols = [col for col in dev_df.columns
                 if col.startswith('answer') or col.startswith('응답')]
print(f"[Valid] 응답 컬럼: {response_cols} ({len(response_cols)}개)")

dev_responses = dev_df[response_cols]
dev_df['pseudo_label'] = dev_responses.mode(axis=1)[0]
dev_df['agreement_rate'] = dev_responses.apply(
    lambda x: (x == dev_df.loc[x.name, 'pseudo_label']).sum() / len(response_cols),
    axis=1
)

print(f"\n=== 일치율(agreement_rate) 분포 ===")
ar_dist = dev_df['agreement_rate'].value_counts().sort_index()
for rate, cnt in ar_dist.items():
    pct = cnt / len(dev_df) * 100
    print(f"  {rate:.0%} 일치: {cnt}개 ({pct:.1f}%)")

AGREEMENT_THRESHOLD = 0.6
dev_filtered = dev_df[dev_df['agreement_rate'] >= AGREEMENT_THRESHOLD].copy()
print(f"\n[Valid 추적 1] 일치율 {AGREEMENT_THRESHOLD:.0%} 이상 필터링 후 남은 데이터 개수: "
      f"{len(dev_filtered)}개 (노이즈 제거로 인한 감소)")
print(f"  필터링으로 제거된 데이터: {len(dev_df) - len(dev_filtered)}개")

test_type_ratios = test_df['q_type'].value_counts(normalize=True)
total_dev_samples = int(len(dev_filtered) * 0.8)

print(f"\n=== Test 질문 유형 비율 (샘플링 목표) ===")
for qtype, ratio in test_type_ratios.items():
    target_n = max(1, int(total_dev_samples * ratio))
    available_n = len(dev_filtered[dev_filtered['q_type'] == qtype])
    status = "✅" if available_n >= target_n else "⚠️ 부족→복원추출"
    print(f"  {qtype:25s}: 비율 {ratio:.1%}, 목표 {target_n}개, 가용 {available_n}개 {status}")

dev_sampled_list = []
for q_type, ratio in test_type_ratios.items():
    n_samples = max(1, int(total_dev_samples * ratio))
    type_data = dev_filtered[dev_filtered['q_type'] == q_type]
    if len(type_data) > 0:
        sampled = resample(
            type_data,
            replace=len(type_data) < n_samples,
            n_samples=n_samples,
            random_state=SEED
        )
        dev_sampled_list.append(sampled)

dev_metadata = pd.concat(dev_sampled_list).reset_index(drop=True)
dev_metadata['answer'] = dev_metadata['pseudo_label']
dev_metadata['apply_aug'] = False

print(f"\n[Valid 추적 2] Test 비율 반영 샘플링 완료 후 전체 Valid 데이터 개수: {len(dev_metadata)}개")
print(f"[Valid 추적 2] Test 비율 반영 샘플링 완료 후 고유 ID 개수: {dev_metadata['id'].nunique()}개")
print("*(참고: Test 분포를 맞추기 위해 특정 유형을 다운/업샘플링하면서 고유 ID가 변동하는 것은 정상입니다)*")

print(f"\n=== Valid 데이터 질문 유형(q_type) 분포 (Test 비율 반영) ===")
print(dev_metadata['q_type'].value_counts(normalize=True).mul(100).round(2).to_string())

print(f"\n[Valid] 최종 일치율 평균: {dev_metadata['agreement_rate'].mean():.2f}")
print(f"[Valid] 최종 일치율 최소: {dev_metadata['agreement_rate'].min():.2f}")

print(f"\n=== dev_metadata DataFrame 구조 ===")
print(f"shape: {dev_metadata.shape}")
print(f"columns: {list(dev_metadata.columns)}")
preview_dev = ['id', 'question', 'answer', 'pseudo_label', 'agreement_rate', 'q_type', 'apply_aug']
print(f"\n처음 3행:\n{dev_metadata[[c for c in preview_dev if c in dev_metadata.columns]].head(3).to_string()}")


========== [Valid] 데이터 처리 ==========

[Valid] 초기 Dev 전체 데이터 개수: 4413개
[Valid] 초기 Dev 고유 ID 개수: 4413개
[Valid] 응답 컬럼: ['answer1', 'answer2', 'answer3', 'answer4', 'answer5'] (5개)

=== 일치율(agreement_rate) 분포 ===
  0% 일치: 1개 (0.0%)
  20% 일치: 1개 (0.0%)
  40% 일치: 903개 (20.5%)
  60% 일치: 2606개 (59.1%)
  80% 일치: 902개 (20.4%)

[Valid 추적 1] 일치율 60% 이상 필터링 후 남은 데이터 개수: 3508개 (노이즈 제거로 인한 감소)
  필터링으로 제거된 데이터: 905개

=== Test 질문 유형 비율 (샘플링 목표) ===
  count                    : 비율 33.8%, 목표 948개, 가용 2433개 ✅
  material                 : 비율 30.5%, 목표 854개, 가용 510개 ⚠️ 부족→복원추출
  object_type              : 비율 17.1%, 목표 481개, 가용 245개 ⚠️ 부족→복원추출
  color                    : 비율 6.4%, 목표 179개, 가용 92개 ⚠️ 부족→복원추출
  recycling_class          : 비율 4.1%, 목표 116개, 가용 51개 ⚠️ 부족→복원추출
  position                 : 비율 4.1%, 목표 115개, 가용 116개 ✅
  dominant_or_majority     : 비율 2.4%, 목표 68개, 가용 33개 ⚠️ 부족→복원추출
  other                    : 비율 0.8%, 목표 22개, 가용 21개 ⚠️ 부족→복원추출
  brand_or_product         : 비율 0.4%, 목표 10개, 가용 4개 ⚠️ 

### 이미지 전처리 함수
여기서는 resize + padding 기준을 정의하고 train/dev 이미지를 실제 모델 입력 형태로 바꾸는 함수를 만듭니다.
작은 물체가 잘리지 않게 정사각 패딩을 쓰는 선택이 코드에 반영되어 있습니다.

In [4]:
# ===== 이미지 전처리 함수 =====
def resize_with_padding(img, target_size=512):
    img = img.copy()
    img.thumbnail((target_size, target_size), Image.LANCZOS)
    padded = Image.new("RGB", (target_size, target_size), (255, 255, 255))
    offset = ((target_size - img.width) // 2, (target_size - img.height) // 2)
    padded.paste(img, offset)
    return padded

# Heavy augmentation (오버샘플된 데이터에만 적용)
heavy_augmentation = A.Compose([
    A.ShiftScaleRotate(
        shift_limit=0.05, scale_limit=0.1, rotate_limit=15,
        border_mode=0, p=0.8
    ),
    A.ColorJitter(
        brightness=0.15, contrast=0.15,
        saturation=0.15, hue=0.05, p=0.6
    ),
    A.CLAHE(clip_limit=2.5, tile_grid_size=(8, 8), p=0.5),
    A.CoarseDropout(
        max_holes=4, max_height=32, max_width=32,
        min_holes=1, fill_value=0, p=0.3
    ),
])

def load_and_preprocess(img_path, apply_aug=False):
    try:
        img = Image.open(img_path).convert("RGB")
        img = resize_with_padding(img, target_size=512)
    except Exception as e:
        print(f"Error loading {img_path}: {e}")
        img = Image.new("RGB", (512, 512), color='white')
    if apply_aug:
        np_img = np.array(img)
        np_img = heavy_augmentation(image=np_img)["image"]
        img = Image.fromarray(np_img)
    return img

def convert_train_image(example):
    img_path = os.path.join(IMAGE_ROOT, example["path"])
    example["decoded_image"] = load_and_preprocess(
        img_path, apply_aug=example.get("apply_aug", False)
    )
    return example

def convert_dev_image(example):
    img_path = os.path.join(IMAGE_ROOT, example["path"])
    example["decoded_image"] = load_and_preprocess(img_path, apply_aug=False)
    return example

# ===== HF Dataset 변환 (Train) =====
raw_train_dataset = Dataset.from_pandas(train_metadata)  # train_df → train_metadata로 변경!
raw_train_dataset = raw_train_dataset.map(convert_train_image, num_proc=os.cpu_count())

# ===== HF Dataset 변환 (Valid) =====
raw_dev_dataset = Dataset.from_pandas(dev_metadata)  # dev_df → dev_metadata로 변경!
raw_dev_dataset = raw_dev_dataset.map(convert_dev_image, num_proc=os.cpu_count())

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_46811/1968173771.py:21: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(


Map (num_proc=48):   0%|          | 0/7764 [00:00<?, ? examples/s]

Map (num_proc=48):   0%|          | 0/2799 [00:00<?, ? examples/s]

### 데이터셋 sanity check
전처리 후 HuggingFace Dataset 길이와 원본 메타데이터 길이가 맞는지 확인합니다.
학습 전에 이미지 누락이나 변환 실패가 없는지 보는 체크 포인트입니다.

In [5]:
# 이미지 로드 검증
print(f"\n========== 이미지 로드 검증 ==========")
print(f"[Train] HF Dataset 크기: {len(raw_train_dataset)} (= train_metadata {len(train_metadata)}와 일치해야 함)")
print(f"[Valid] HF Dataset 크기: {len(raw_dev_dataset)} (= dev_metadata {len(dev_metadata)}와 일치해야 함)")
assert len(raw_train_dataset) == len(train_metadata), "❌ Train Dataset 크기 불일치!"
assert len(raw_dev_dataset) == len(dev_metadata), "❌ Valid Dataset 크기 불일치!"
print("✅ Dataset 크기 검증 통과")


========== 이미지 로드 검증 ==========
[Train] HF Dataset 크기: 7764 (= train_metadata 7764와 일치해야 함)
[Valid] HF Dataset 크기: 2799 (= dev_metadata 2799와 일치해야 함)
✅ Dataset 크기 검증 통과


## 4. Unsloth 모델 및 LoRA 어댑터 로딩

In [6]:
'''
LoRA가 학습할 수 있는 파라미터가 많아져서 더 복잡한 패턴을 학습할 수 있음.
단점: VRAM 사용량이 늘어나고, 데이터가 적으면 과적합 위험

현재: alpha=32, r=32 → 비율 1.0
만약: alpha=64, r=32 → 비율 2.0 (LoRA 영향이 2배)
만약: alpha=32, r=64 → 비율 0.5 (LoRA 영향이 절반)

[다음 목표] 플랜C
데이터 규모           r       alpha     이유
5000개 + 성능 부족 시 64      64        학습 용량 늘림, 비율 1.0 유지
'''
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    dtype = torch.bfloat16,
    fast_inference = False,
    gpu_memory_utilization = 0.8,
    attn_implementation =  "sdpa", # "flash_attention_2", # Flash Attention 명시적 적용
)

# LoRA 어댑터 추가 (업데이트된 설정)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 32,           # Increased from 16
    lora_alpha = 32,  # Increased from 16
    lora_dropout = 0.05,# Added dropout
    bias = "none",
    random_state = SEED,
    use_rslora = True,  # Enabled Rank Stabilized LoRA
    loftq_config = None,
    use_gradient_checkpointing = "unsloth",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Explicitly target modules
)

model.print_trainable_parameters()

==((====))==  Unsloth 2026.4.2: Fast Qwen3_Vl patching. Transformers: 4.57.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


trainable params: 268,435,456 || all params: 33,625,825,520 || trainable%: 0.7983


## 5. 프롬프트 템플릿 및 데이터 포맷팅

In [7]:
# 프롬프트: 재활용품 도메인 특화 영어 구조화 프롬프트
SYSTEM_INSTRUCT = (
    "You are an expert visual analysis AI specialized in recycling and waste classification.\n"
    "Your capabilities include:\n"
    "1. Precise identification of recyclable materials (plastic, glass, metal, paper, etc.)\n"
    "2. Counting and distinguishing items in waste/recycling images\n"
    "3. Recognizing packaging types, labels, and material compositions\n\n"
    "OUTPUT REQUIREMENTS:\n"
    "- Return exactly one lowercase letter (a, b, c, or d)\n"
    "- No other text, punctuation, or explanations\n\n"
    "QUALITY STANDARDS:\n"
    "1. Examine all visual details thoroughly, including material textures and labels\n"
    "2. Consider the specific context of recycling classification standards\n"
    "3. Evaluate all options systematically\n"
    "4. Choose the single most accurate answer based on visual evidence"
)

def build_mc_prompt(question, a, b, c, d):
    return (
        "TASK: Recycling & Waste Visual Question Analysis\n\n"
        f"QUESTION TO ANALYZE:\n{question}\n\n"
        f"OPTIONS TO EVALUATE:\n"
        f"a) {a}\nb) {b}\nc) {c}\nd) {d}\n\n"
        "ANALYSIS STEPS:\n"
        "1. Examine all visual elements in the image (materials, labels, shapes, colors)\n"
        "2. Understand the specific requirements of the question\n"
        "3. Consider each option against the visual evidence\n"
        "4. Select the most accurate answer\n\n"
        "RESPONSE FORMAT:\n"
        "Provide exactly one lowercase letter (a, b, c, or d)."
    )

# 학습 데이터를 모델의 대화 형식(messages)으로 변환하는 함수
def make_conversation(example):
    user_text = build_mc_prompt(str(example["question"]), str(example["a"]), str(example["b"]), str(example["c"]), str(example["d"]))
    gold = str(example["answer"]).strip().lower()

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {
            "role":"user",
            "content":[
                {"type":"image"},
                {"type":"text","text":user_text}
                ]
         },
          {
              "role":"assistant",
              "content":[
                  {"type":"text","text":gold}
                  ]
            }
    ]
    image_data = [example["decoded_image"]] if "decoded_image" in example else []
    return {"messages": messages, "images": image_data}

remove_cols = [
    "path", "a", "b", "c", "d", "question", "id", "answer", "decoded_image",
    "apply_aug", "material_class", "q_type",
    "pseudo_label", "agreement_rate", "weight",
]

# Train
train_dataset = raw_train_dataset.map(
    make_conversation,
    remove_columns=[c for c in remove_cols if c in raw_train_dataset.column_names],
    num_proc=os.cpu_count()
)

# Valid — dev에는 answer1~5 등 추가 컬럼이 남아있을 수 있음
dev_extra_cols = [c for c in raw_dev_dataset.column_names
                  if (c.startswith('answer') or c.startswith('응답'))
                  and c not in remove_cols]
valid_remove = [c for c in remove_cols if c in raw_dev_dataset.column_names] + dev_extra_cols

valid_dataset = raw_dev_dataset.map(
    make_conversation,
    remove_columns=valid_remove,
    num_proc=os.cpu_count()
)

# 잔여 컬럼 검증
expected_cols = {'messages', 'images'}
train_extra = set(train_dataset.column_names) - expected_cols
valid_extra = set(valid_dataset.column_names) - expected_cols
if train_extra:
    print(f"⚠️ Train에 불필요한 잔여 컬럼: {train_extra}")
if valid_extra:
    print(f"⚠️ Valid에 불필요한 잔여 컬럼: {valid_extra}")
if not train_extra and not valid_extra:
    print("✅ 잔여 컬럼 없음 — messages, images만 남아있음")

Map (num_proc=48):   0%|          | 0/7764 [00:00<?, ? examples/s]

Map (num_proc=48):   0%|          | 0/2799 [00:00<?, ? examples/s]

✅ 잔여 컬럼 없음 — messages, images만 남아있음


### 프롬프트 길이 점검
모델 입력 텍스트 길이를 미리 세어 너무 긴 샘플이 없는지 확인합니다.
max length, 배치 크기, 메모리 사용량을 조정할 때 참고하는 셀입니다.

In [8]:
# ==========================================
# 시퀀스 길이 확인 (학습 전에 실행)
# ==========================================

from collections import Counter

lengths = []
over_1024 = 0
over_2048 = 0

# train_dataset에서 샘플링해서 확인
for i in tqdm(range(len(raw_train_dataset)), desc="토큰 길이 측정"):
    example = raw_train_dataset[i]

    img_path = os.path.join(IMAGE_ROOT, example["path"])
    try:
        image = Image.open(img_path).convert("RGB")
        image = resize_with_padding(image, 512)
    except:
        image = Image.new("RGB", (512, 512), color='white')

    user_text = build_mc_prompt(
        str(example["question"]),
        str(example["a"]), str(example["b"]),
        str(example["c"]), str(example["d"])
    )

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]},
        {"role": "assistant", "content": [{"type": "text", "text": "a"}]},
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
    )

    inputs = tokenizer(
        image, prompt,
        add_special_tokens=False,
        return_tensors="pt",
    )

    seq_len = inputs["input_ids"].shape[1]
    lengths.append(seq_len)

    if seq_len > 1024:
        over_1024 += 1
    if seq_len > 2048:
        over_2048 += 1

print(f"\n========== 시퀀스 길이 통계 ==========")
print(f"샘플 수:  {len(lengths)}")
print(f"최소:     {min(lengths)}")
print(f"최대:     {max(lengths)}")
print(f"평균:     {sum(lengths)/len(lengths):.0f}")
print(f"중앙값:   {sorted(lengths)[len(lengths)//2]}")
print(f"\n1024 초과: {over_1024}개 ({over_1024/len(lengths)*100:.1f}%)")
print(f"2048 초과: {over_2048}개 ({over_2048/len(lengths)*100:.1f}%)")

# 분포 구간별 확인
print(f"\n=== 구간별 분포 ===")
bins = [0, 256, 512, 768, 1024, 1536, 2048, float('inf')]
labels = ["~256", "257~512", "513~768", "769~1024", "1025~1536", "1537~2048", "2048~"]
for j in range(len(bins)-1):
    cnt = sum(1 for l in lengths if bins[j] < l <= bins[j+1])
    pct = cnt / len(lengths) * 100
    bar = "█" * int(pct / 2)
    print(f"  {labels[j]:>10s}: {cnt:5d}개 ({pct:5.1f}%) {bar}")

토큰 길이 측정:  26%|██▌       | 2036/7764 [00:40<01:53, 50.52it/s]


KeyboardInterrupt: 

### DataLoader worker 수 확인
현재 런타임에서 실제로 쓸 수 있는 CPU 수를 확인해 DataLoader 병렬 설정에 참고합니다.
Colab/로컬 환경 차이 때문에 이 값을 따로 찍어보는 흐름입니다.

In [9]:
import multiprocessing

print(f"os.cpu_count():              {os.cpu_count()}")
print(f"multiprocessing.cpu_count(): {multiprocessing.cpu_count()}")

# 실제 사용 가능한 CPU (컨테이너 환경에서는 이게 더 정확)
try:
    available = len(os.sched_getaffinity(0))
    print(f"os.sched_getaffinity(0):     {available}")
except AttributeError:
    pass

!nproc

os.cpu_count():              48
multiprocessing.cpu_count(): 48
os.sched_getaffinity(0):     48
48


## 6. SFTTrainer를 사용한 최종 파인튜닝

### 최종 학습 설정과 실행
여기서 batch size, gradient accumulation, epoch, early stopping, wandb 로깅 같은 핵심 학습 설정을 묶어 SFTTrainer를 실행합니다.
즉, 이 노트북의 실제 파인튜닝 본체입니다.

In [10]:
from transformers import EarlyStoppingCallback  # ← 추가
# 모델을 학습 모드로 활성화
FastVisionModel.for_training(model)

# 플랜A (1)(2): 배치 크기 및 학습 설정 변경
per_device_train_batch_size = 24   # 8 → 2 (32B 모델 VRAM 대응)
gradient_accumulation_steps = 1   # 2 → 8 (effective batch size 16 유지)
effective_batch_size = per_device_train_batch_size * gradient_accumulation_steps
num_train_epochs = 5              # 5 → 1 (과적합 방지)
learning_rate = 1e-4              # Unsloth 권장 범위 5e-5 ~ 1e-4
total_train_steps = math.ceil((len(train_dataset) * num_train_epochs) / effective_batch_size)
eval_steps = math.ceil(len(train_dataset) / effective_batch_size / 2) # Evaluate twice per epoch
logging_steps = 10 # Log every 10 steps

print(f"Effective Batch Size: {effective_batch_size}")
print(f"Total Training Steps: {total_train_steps}")
print(f"Evaluation Steps: {eval_steps}")

# SFTTrainer 설정 (플랜A 적용)
trainer = SFTTrainer(
    model=model,
    dataset_text_field="", # (Vision fine-tuning 필수)
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    packing=False,  # Vision 모델에서 packing은 이미지 토큰 정렬 문제 유발
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # ← 추가
    args=SFTConfig(
        dataloader_num_workers=8,       # ← 추가
        dataloader_pin_memory=True,     # ← 추가 (GPU 전송 속도 향상)
        dataloader_prefetch_factor=2,   # ← 추가 (미리 다음 배치 준비)
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_train_batch_size * 2,
        gradient_accumulation_steps=gradient_accumulation_steps,
        warmup_steps=int(total_train_steps * 0.05),
        num_train_epochs=num_train_epochs,
        learning_rate=learning_rate, # Unsloth 권장 범위 5e-5 ~ 1e-4
        logging_steps=logging_steps,
        eval_strategy="steps",
        eval_steps=eval_steps,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=SEED,
        output_dir=SAVE_DIR,
        report_to="wandb",
        run_name=RUN_NAME,
        remove_unused_columns=False,
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=MAX_SEQ_LENGTH,
        save_strategy="steps",
        save_steps=eval_steps,
        save_total_limit=5,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        dataloader_drop_last=False,
    ),
)
print(f"현재 할당: {torch.cuda.memory_allocated()/1024**3:.1f}GB")
print(f"최대 피크: {torch.cuda.max_memory_allocated()/1024**3:.1f}GB")
print(f"예약:     {torch.cuda.memory_reserved()/1024**3:.1f}GB")

# 학습 시작
print("Starting final training...")

# 혹시 체크포인트가 깨졌을 때
# import shutil
# shutil.rmtree("outputs_v3/checkpoint-XXX")  # 깨진 것 삭제
# trainer.train(resume_from_checkpoint=True)  # 최신 checkpoint 자동 탐색
trainer.train()
print("Final training finished.")

# 최종 평가
print("Evaluating the best model...")
final_eval_results = trainer.evaluate()
print("Final Evaluation Results:", final_eval_results)

# Wandb 종료
wandb.finish()

Effective Batch Size: 24
Total Training Steps: 1618
Evaluation Steps: 162
Unsloth: Model does not have a default image size - using 512
현재 할당: 31.1GB
최대 피크: 31.1GB
예약:     31.1GB
Starting final training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,764 | Num Epochs = 5 | Total steps = 1,620
O^O/ \_/ \    Batch size per device = 24 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (24 x 1 x 1) = 24
 "-____-"     Trainable parameters = 268,435,456 of 33,625,825,520 (0.80% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
162,0.056700,0.071413
324,0.048300,0.070246


Step,Training Loss,Validation Loss
162,0.056700,0.071413
324,0.048300,0.070246
486,0.038300,0.071471


KeyboardInterrupt: 

## 7. 추론 및 제출 파일 생성

In [ ]:
!ls -la ./{SAVE_DIR}/

### 체크포인트 로드 후 추론
학습이 끝난 뒤 저장된 체크포인트를 다시 불러와 test.csv 전체에 대해 추론하고 submission을 만듭니다.
단일 노트북 안에서 학습과 제출 생성까지 한 번에 마무리하는 구성입니다.

In [11]:
import os, gc, glob
from PIL import Image
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from unsloth import FastVisionModel
from transformers import LogitsProcessorList

# 0) 메모리 해제 (학습 모델이 올라가 있을 경우)
try:
    del model, tokenizer, trainer
    gc.collect()
    torch.cuda.empty_cache()
except:
    pass

# 1) 모델 로드
CKPT_DIR = sorted(glob.glob(f"./{SAVE_DIR}/checkpoint-*"))[-1]
print(f"Loading: {CKPT_DIR}")

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=CKPT_DIR,
    load_in_4bit=True,
    dtype=torch.float16,
    device_map="auto",
)
FastVisionModel.for_inference(model)

# 2) 토큰 ID 설정 (기존과 동일)
def get_text_tokenizer(tkr):
    if hasattr(tkr, "tokenizer") and tkr.tokenizer is not None:
        return tkr.tokenizer
    if hasattr(tkr, "text_tokenizer") and tkr.text_tokenizer is not None:
        return tkr.text_tokenizer
    return tkr

txt_tok = get_text_tokenizer(tokenizer)

def gather_single_token_ids(txt_tok, ch):
    candidates = [ch, " " + ch, ch.lower(), " " + ch.lower(), ch.upper(), " " + ch.upper()]
    ids = set()
    for s in candidates:
        try:
            toks = txt_tok.encode(s, add_special_tokens=False)
        except TypeError:
            toks = txt_tok.encode(s)
        if isinstance(toks, dict) and "input_ids" in toks:
            toks = toks["input_ids"]
        if isinstance(toks, (list, tuple)) and len(toks) == 1:
            ids.add(int(toks[0]))
    return list(ids)

A_IDS = gather_single_token_ids(txt_tok, "a")
B_IDS = gather_single_token_ids(txt_tok, "b")
C_IDS = gather_single_token_ids(txt_tok, "c")
D_IDS = gather_single_token_ids(txt_tok, "d")

class RestrictToSet:
    def __init__(self, allowed_ids):
        self.allowed = torch.tensor(sorted(list(allowed_ids)))
    def __call__(self, input_ids, scores):
        mask = scores.new_full(scores.shape, float("-inf"))
        mask[:, self.allowed] = scores[:, self.allowed]
        return mask

allowed_all = set(A_IDS + B_IDS + C_IDS + D_IDS)
logits_processors = LogitsProcessorList([RestrictToSet(allowed_all)]) if len(allowed_all) > 0 else None

# 3) 경로 설정
SAVE_PATH = f"{SAVE_DIR}/Q3-32B-Ins-ep5-submission.csv"
TEMP_SAVE_PATH = f"{SAVE_DIR}/Q3-32B-Ins-ep5-submission_temp.csv"

ENSEMBLE_PATH = f"{SAVE_DIR}/Q3-32B-Ins-ep5-ensemble.csv"  # ← 추가
TEMP_ENSEMBLE_PATH = f"{SAVE_DIR}/Q3-32B-Ins-ep5-ensemble_temp.csv"  # ← 추가

SAVE_EVERY = 100
MAX_NEW_TOKENS = 1

# 4) 중간 저장 이어하기
preds = []
start_idx = 0

if os.path.exists(TEMP_SAVE_PATH):
    temp_df = pd.read_csv(TEMP_SAVE_PATH)
    preds = temp_df.to_dict('records')
    start_idx = len(preds)
    # 앙상블 파일도 같이 복원
    if os.path.exists(TEMP_ENSEMBLE_PATH):
        preds_with_probs = pd.read_csv(TEMP_ENSEMBLE_PATH).to_dict('records')
    print(f"✅ 중간 저장 파일 발견: {start_idx}개 완료분 로드, {start_idx}번부터 이어서 추론")
else:
    print("중간 저장 파일 없음 — 처음부터 시작")

# 5) 추론 (1개씩 — VLM은 이미지 크기가 달라 배치가 어려움)
model.eval()
with torch.inference_mode():
    for i in tqdm(range(start_idx, len(test_df)), initial=start_idx, total=len(test_df)):
        row = test_df.iloc[i]
        img_path = os.path.join(IMAGE_ROOT, str(row["path"]))
        image = Image.open(img_path).convert("RGB")

        user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]},
        ]

        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
        inputs = tokenizer(
            image, prompt,
            add_special_tokens=False, return_tensors="pt",
        ).to("cuda")

        gen_kwargs = dict(
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, temperature=0.0, top_p=1.0,
            use_cache=True, return_dict_in_generate=True, output_scores=True,
        )
        if logits_processors is not None:
            gen_kwargs["logits_processor"] = logits_processors

        outputs = model.generate(**inputs, **gen_kwargs)

        scores = outputs.scores[-1][0]
        probs = F.softmax(scores, dim=-1)

        def prob_of(ids):
            return float(probs[ids].sum().item()) if ids else -1.0

        probs_dict = {"a": prob_of(A_IDS), "b": prob_of(B_IDS),
                      "c": prob_of(C_IDS), "d": prob_of(D_IDS)}
        pred = max(probs_dict, key=probs_dict.get)

        preds.append({"id": row["id"], "answer": pred})

        # ← 추가: 확률도 저장
        preds_with_probs.append({
            "id": row["id"],
            "answer": pred,
            "prob_a": round(probs_dict["a"], 6),
            "prob_b": round(probs_dict["b"], 6),
            "prob_c": round(probs_dict["c"], 6),
            "prob_d": round(probs_dict["d"], 6),
        })

        # 중간 저장
        if len(preds) % SAVE_EVERY == 0:
            pd.DataFrame(preds, columns=["id", "answer"]).to_csv(TEMP_SAVE_PATH, index=False)
            pd.DataFrame(preds_with_probs).to_csv(TEMP_ENSEMBLE_PATH, index=False)  # ← 추가
            print(f"\n💾 중간 저장: {len(preds)}/{len(test_df)} 완료")

# 최종 저장
submission = pd.DataFrame(preds, columns=["id", "answer"])
submission.to_csv(SAVE_PATH, index=False)
print(f"\nSaved {SAVE_PATH}")
print(submission.head())

# 앙상블 파일 저장
ensemble = pd.DataFrame(preds_with_probs)
ensemble.to_csv(ENSEMBLE_PATH, index=False)
print(f"Saved {ENSEMBLE_PATH}")
print(ensemble.head())
print(f"\n답변 분포:\n{submission['answer'].value_counts()}")

# # 임시 파일 삭제
# for tmp in [TEMP_SAVE_PATH, TEMP_ENSEMBLE_PATH]:
#     if os.path.exists(tmp):
#         os.remove(tmp)
# print("🗑️ 임시 저장 파일 삭제 완료")

Loading: ./run_outputs/(valid)qwen3_v2_32B_epoch5_standard/checkpoint-486
==((====))==  Unsloth 2026.4.2: Fast Qwen3_Vl patching. Transformers: 4.57.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

✅ 중간 저장 파일 발견: 100개 완료분 로드, 100번부터 이어서 추론


  4%|▍         | 200/5074 [00:54<1:03:28,  1.28it/s]


💾 중간 저장: 200/5074 완료


...


💾 중간 저장: 4900/5074 완료


 99%|█████████▊| 5000/5074 [1:00:17<00:56,  1.31it/s]


💾 중간 저장: 5000/5074 완료


100%|██████████| 5074/5074 [1:01:13<00:00,  1.35it/s]


Saved run_outputs/(valid)qwen3_v2_32B_epoch5_standard/Q3-32B-Ins-ep5-submission.csv
              id answer
0  test_0001.jpg      d
1  test_0002.jpg      b
2  test_0003.jpg      c
3  test_0004.jpg      a
4  test_0005.jpg      b

답변 분포:
answer
c    1303
d    1280
b    1264
a    1227
Name: count, dtype: int64


In [ ]:
# 모든 작업 완료 후
print("Job Done! Terminating session in 30 seconds...")
import time
from google.colab import runtime

time.sleep(30)
runtime.unassign()

time.sleep(10)
os.kill(os.getpid(), 9)

Job Done! Terminating session in 30 seconds...


### 선택 작업 메모
마지막 셀들은 필수 실행 구간이 아니라 체크포인트 압축, Drive 저장, LoRA 어댑터 저장 같은 선택 작업을 메모해 둔 부분입니다.
재실행 시에는 필요한 것만 골라 쓰면 됩니다.

In [ ]:
# !zip -r /content/outputs_final.zip /content/outputs_final/checkpoint-318

In [ ]:
# 나의 구글 드라이브 본래 작업 폴더에 저장
# drive_path = "/content/drive/My Drive/251024/submission_final.csv" # 경로 확인 필요, 파일명 변경
# submission.to_csv(drive_path, index=False)
# print(f"Saved to Google Drive: {drive_path}")

In [ ]:
# 최종 학습된 LoRA 어댑터 저장 (옵션)
# model.save_pretrained("final_model_lora")
# tokenizer.save_pretrained("final_model_lora")
# print("Saved LoRA adapter and tokenizer to 'final_model_lora'")